# 🚀 Envoy Proxy デモ on Google Colab

**Envoy バイナリを公式 apt リポジトリから直接インストール**して以下をデモします：

1. **基本的な HTTP プロキシ** — リクエストをバックエンドに転送
2. **ロードバランシング** — 複数のバックエンドにラウンドロビン分散
3. **管理 API** — クラスター状態・統計情報の確認
4. **フェイルオーバー** — バックエンド障害時の自動切り替え

```
Client (Python)
      │
      ▼
┌─────────────┐
│ Envoy :10000│  ← フロントプロキシ
└──────┬──────┘
       │  ラウンドロビン
  ┌────┴────┐
  ▼         ▼
:8081    :8082   ← Python バックエンドサーバー
```

## 1. Envoy バイナリをインストール

GitHub Releases から Linux 用バイナリを直接ダウンロードします。
apt リポジトリ不要で確実に動作します。

In [ ]:
ENVOY_VERSION = "1.30.1"

!apt-get install -y -qq curl
!curl -L https://github.com/envoyproxy/envoy/releases/download/v{ENVOY_VERSION}/envoy-{ENVOY_VERSION}-linux-x86_64 \
  -o /usr/local/bin/envoy
!chmod +x /usr/local/bin/envoy

!envoy --version
print("✅ Envoy インストール完了")

## 2. バックエンドサーバーを起動

Python で 2 台の HTTP サーバーを起動します。
それぞれ「どのサーバーが応答したか」を JSON で返します。

In [ ]:
import subprocess, time, os, json, urllib.request

# 既存プロセスをクリーンアップ
for port in [8081, 8082, 10000, 9901]:
    subprocess.run(f"fuser -k {port}/tcp 2>/dev/null", shell=True)
time.sleep(1)

backend_procs = []

for port, name in [(8081, "Backend-A"), (8082, "Backend-B")]:
    script = f"""\
from http.server import HTTPServer, BaseHTTPRequestHandler
import json, datetime
class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        body = json.dumps({{
            "server": "{name}",
            "port": {port},
            "path": self.path,
            "time": datetime.datetime.now().isoformat()
        }}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, *a): pass
HTTPServer(('127.0.0.1', {port}), Handler).serve_forever()
"""
    path = f"/tmp/backend_{port}.py"
    with open(path, "w") as f:
        f.write(script)
    proc = subprocess.Popen(["python3", path])
    backend_procs.append(proc)
    print(f"✅ {name} を :{port} で起動  (PID {proc.pid})")

time.sleep(1)

print("\n--- バックエンド直接確認 ---")
for port in [8081, 8082]:
    res = urllib.request.urlopen(f"http://127.0.0.1:{port}/health")
    print(json.loads(res.read()))

## 3. Envoy 設定ファイルを作成

- `:10000` でリクエストを受け付け
- Backend-A / B に **ROUND_ROBIN** で転送
- `:9901` で管理 API を公開

In [ ]:
envoy_config = """\
static_resources:
  listeners:
  - name: listener_0
    address:
      socket_address:
        address: 127.0.0.1
        port_value: 10000
    filter_chains:
    - filters:
      - name: envoy.filters.network.http_connection_manager
        typed_config:
          "@type": type.googleapis.com/envoy.extensions.filters.network.http_connection_manager.v3.HttpConnectionManager
          stat_prefix: ingress_http
          access_log:
          - name: envoy.access_loggers.file
            typed_config:
              "@type": type.googleapis.com/envoy.extensions.access_loggers.file.v3.FileAccessLog
              path: /tmp/envoy_access.log
          route_config:
            name: local_route
            virtual_hosts:
            - name: local_service
              domains: ["*"]
              routes:
              - match:
                  prefix: "/"
                route:
                  cluster: backend_cluster
          http_filters:
          - name: envoy.filters.http.router
            typed_config:
              "@type": type.googleapis.com/envoy.extensions.filters.http.router.v3.Router

  clusters:
  - name: backend_cluster
    connect_timeout: 1s
    type: STATIC
    lb_policy: ROUND_ROBIN
    load_assignment:
      cluster_name: backend_cluster
      endpoints:
      - lb_endpoints:
        - endpoint:
            address:
              socket_address:
                address: 127.0.0.1
                port_value: 8081
        - endpoint:
            address:
              socket_address:
                address: 127.0.0.1
                port_value: 8082

admin:
  address:
    socket_address:
      address: 127.0.0.1
      port_value: 9901
"""

with open("/tmp/envoy.yaml", "w") as f:
    f.write(envoy_config)

print("✅ /tmp/envoy.yaml を保存")
print(envoy_config)

## 4. Envoy をバックグラウンドで起動

In [ ]:
envoy_log = open("/tmp/envoy.log", "w")
envoy_proc = subprocess.Popen(
    ["envoy", "-c", "/tmp/envoy.yaml", "--log-level", "warn"],
    stdout=envoy_log,
    stderr=envoy_log
)
print(f"Envoy 起動中... (PID {envoy_proc.pid})")
time.sleep(3)

if envoy_proc.poll() is None:
    print("✅ Envoy 起動成功")
    print("   プロキシ : http://127.0.0.1:10000")
    print("   管理API  : http://127.0.0.1:9901")
else:
    print("❌ 起動失敗。ログ:")
    with open("/tmp/envoy.log") as f:
        print(f.read()[-3000:])

## 5. ロードバランシングのデモ

Envoy に 10 回リクエストを送り、**ラウンドロビンで均等分散**されることを確認します。

In [ ]:
print("=" * 52)
print(" Envoy (:10000) へ 10 リクエスト送信")
print("=" * 52)

counter = {"Backend-A": 0, "Backend-B": 0}

for i in range(1, 11):
    res = urllib.request.urlopen("http://127.0.0.1:10000/hello")
    data = json.loads(res.read())
    counter[data["server"]] += 1
    icon = "🅰️ " if data["server"] == "Backend-A" else "🅱️ "
    print(f"  [{i:02d}] {icon} {data['server']}  (port {data['port']})")

print()
print("=" * 52)
print(" 📊 集計")
for k, v in counter.items():
    bar = "█" * v + "░" * (10 - v)
    print(f"  {k}: {bar}  {v}/10 回")
print("=" * 52)
print("✅ ラウンドロビンで均等に分散されました！")

## 6. 管理 API で統計情報を確認

In [ ]:
res = urllib.request.urlopen("http://127.0.0.1:9901/stats?filter=backend_cluster")
stats = res.read().decode()

print("=" * 60)
print(" 📈 Envoy 統計情報 (backend_cluster)")
print("=" * 60)

show_keys = [
    "upstream_cx_total",
    "upstream_cx_active",
    "upstream_rq_total",
    "upstream_rq_completed",
    "upstream_rq_2xx",
    "membership_total",
    "membership_healthy",
]
for line in stats.splitlines():
    for k in show_keys:
        if f".{k}: " in line or line.endswith(f".{k}: 0"):
            label = line.rsplit(".", 1)[-1].split(": ")[0]
            value = line.split(": ")[-1]
            print(f"  {label:<35} {value}")
            break

In [ ]:
print("=" * 60)
print(" 📝 Envoy アクセスログ (最新 10 件)")
print("=" * 60)
try:
    with open("/tmp/envoy_access.log") as f:
        lines = f.readlines()
    for line in lines[-10:]:
        print(" ", line.rstrip())
except FileNotFoundError:
    print("(ログファイルがまだ生成されていません)")

## 7. フェイルオーバーのデモ

Backend-A を停止して、Envoy が自動的に Backend-B のみに転送することを確認します。

In [ ]:
backend_procs[0].terminate()
backend_procs[0].wait()
print(f"🛑 Backend-A (PID {backend_procs[0].pid}) を停止")
time.sleep(1)

print()
print("=" * 52)
print(" Backend-A 停止後: 6 リクエスト送信")
print("=" * 52)

for i in range(1, 7):
    try:
        res = urllib.request.urlopen("http://127.0.0.1:10000/test", timeout=3)
        data = json.loads(res.read())
        print(f"  [{i}] ✅ {data['server']} (port {data['port']})")
    except Exception as e:
        print(f"  [{i}] ⚠️  {type(e).__name__}: {e}")

print()
print("✅ Backend-B のみで応答を継続")

## 8. クリーンアップ

In [ ]:
envoy_proc.terminate()
for p in backend_procs:
    try: p.terminate()
    except: pass
envoy_log.close()
print("✅ 全プロセスを停止しました")

---
## まとめ

| デモした機能 | 内容 |
|---|---|
| **HTTP プロキシ** | Envoy (:10000) がリクエストをバックエンドに転送 |
| **ロードバランシング** | ROUND_ROBIN で Backend-A / B に均等分散 |
| **管理 API** | `/stats` でリアルタイムにメトリクスを確認 |
| **フェイルオーバー** | Backend-A 停止後も Backend-B が応答を継続 |

### 次のステップ
- **ヘルスチェック**: `health_checks` でバックエンドを自動監視
- **レート制限**: `envoy.filters.http.local_ratelimit` で過負荷保護
- **TLS 終端**: `tls_context` で HTTPS 対応
- **観測性**: Prometheus + Grafana でメトリクス可視化

> 📖 公式ドキュメント: https://www.envoyproxy.io/docs